# CASMI 2026 — Piloto B: decodificador huella Morgan → SMILES

**Experimento exploratorio**, complementario al retrieval RGB y no sustituto del envío.
El piloto obtuvo MRR@25 0.3372 en GEN-VAL usando huellas exactas, pero no midió novedad y no integra espectro→huella. Además, el exact-match greedy del holdout fue 0.232 (puerta propuesta: 0.50). No está listo para envío.

Familia B del protocolo: etapa 1 (espectro → huella) + etapa 2 (huella → SMILES). Este notebook solo entrena y evalúa la etapa 2. Arquitectura: Transformer encoder-decoder pequeño desde cero.

In [ ]:
# Celda 1 — GPU y versiones (este notebook exige runtime GPU).
import sys
import torch
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| memoria:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
import numpy, scipy
print("numpy", numpy.__version__, "| scipy", scipy.__version__)
assert torch.cuda.is_available(), (
    "Este notebook exige runtime GPU: Entorno > Cambiar tipo de entorno > Acelerador de hardware > GPU.")


## Paquete de datos

El notebook necesita el **paquete de datos** (preparado en la Fase 0,
`verificacion/piloto_b/` del repo local), con la MISMA estructura de
carpetas del repo:

```
datos/
├── entrenar_decodificador.py
├── 30_evaluar_generativo.py
├── verificacion/
│   ├── split_arrays.npz
│   └── piloto_b/
│       ├── gen_val_verdad.parquet
│       ├── auditoria_fuga.json
│       └── colisiones_huella.json
└── intermedios/
    ├── mapa_smiles_full.parquet
    └── huellas_pool.npz
```

Dos opciones para ponerlo en Colab:
- **A (simple):** sube la carpeta `datos/` a `/content/datos` (el
  contenido se pierde al reiniciar el runtime; sirve para una corrida).
- **B (recomendado):** monta Drive y pon el paquete en
  `Mi unidad/casmi-piloto-b/datos/` (persiste entre sesiones).

La siguiente celda monta Drive (si falla, se sigue con la opción A) y
localiza el paquete en cualquiera de las dos rutas.

Antes de subir el paquete, sustituye sus dos scripts por las versiones
versionadas en `codigo/piloto_b/` (`entrenar_decodificador.py` y
`30_evaluar_generativo.py`). El ZIP original contiene la versión previa;
no ejecutes el ZIP sin actualizar esos archivos.


In [ ]:
# Celda 2 — Monta Drive (opcional) y localiza el paquete de datos.
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive montado.")
except Exception as exc:  # fuera de Colab o sin permiso
    print("Drive no montado (se sigue con subida directa):", exc)

CANDIDATOS = [Path("/content/datos"),
              Path("/content/drive/MyDrive/casmi-piloto-b/datos")]
RUTA_DATOS = next((p for p in CANDIDATOS
                   if (p / "verificacion" / "split_arrays.npz").exists()), None)
assert RUTA_DATOS is not None, (
    "No se encontró el paquete de datos en /content/datos ni en "
    "Drive/casmi-piloto-b/datos. Sube el paquete (ver README) y re-corre.")
RUTA_TRABAJO = Path("/content/piloto_b")
RUTA_TRABAJO.mkdir(parents=True, exist_ok=True)
print("Paquete de datos en:", RUTA_DATOS)
print("Trabajo en:", RUTA_TRABAJO)


In [ ]:
# Celda 3 — Verifica que el paquete de datos esté completo.
ARCHIVOS = {
    "entrenar_decodificador.py": RUTA_DATOS / "entrenar_decodificador.py",
    "30_evaluar_generativo.py": RUTA_DATOS / "30_evaluar_generativo.py",
    "verificacion/split_arrays.npz": RUTA_DATOS / "verificacion" / "split_arrays.npz",
    "intermedios/mapa_smiles_full.parquet": RUTA_DATOS / "intermedios" / "mapa_smiles_full.parquet",
    "intermedios/huellas_pool.npz": RUTA_DATOS / "intermedios" / "huellas_pool.npz",
    "verificacion/piloto_b/gen_val_verdad.parquet": RUTA_DATOS / "verificacion" / "piloto_b" / "gen_val_verdad.parquet",
    "verificacion/piloto_b/auditoria_fuga.json": RUTA_DATOS / "verificacion" / "piloto_b" / "auditoria_fuga.json",
    "verificacion/piloto_b/colisiones_huella.json": RUTA_DATOS / "verificacion" / "piloto_b" / "colisiones_huella.json",
}
for nombre, ruta in ARCHIVOS.items():
    assert ruta.exists(), f"FALTA {nombre} en {RUTA_DATOS} — revisa el paquete de datos."
    print(f"OK   {nombre:55s} {ruta.stat().st_size / 1e6:8.1f} MB")
print("Paquete de datos completo.")


In [ ]:
# Celda 4 — Instala dependencias (torch ya viene en Colab).
# RDKit: el scorer usa la versión pinned 2026.03.3; en Colab se instala
# la disponible en PyPI y se reporta. Si difiere de 2026.03.3, esta
# evaluación es una PRIMERA MEDIDA y la final se re-corre con la versión
# pinned (la canonicalización de tautómeros puede mover ~1.6% de claves,
# protocolo §7).
!pip install -q rdkit scipy scikit-learn pyarrow
import rdkit
print("rdkit instalado:", rdkit.__version__)


In [ ]:
# Celda 5 — Carga y verifica los datos + números de la Fase 0.
import json
import numpy as np
col = json.loads((RUTA_DATOS / "verificacion" / "piloto_b" / "colisiones_huella.json").read_text())
aud = json.loads((RUTA_DATOS / "verificacion" / "piloto_b" / "auditoria_fuga.json").read_text())
print("Fase 0 — colisiones de huella (techo familia B):")
print(json.dumps(col, indent=2, ensure_ascii=False))
print()
print("Fase 0 — auditoría de fuga:")
print(json.dumps(aud, indent=2, ensure_ascii=False))
print()
split = np.load(RUTA_DATOS / "verificacion" / "split_arrays.npz", allow_pickle=True)
print(f"splits: {len(split['mols']):,} moléculas; val {len(split['val_mols']):,}; "
      f"spec {len(split['spec_mols']):,} (GEN-TRAIN = "
      f"{len(split['mols']) - len(split['val_mols']) - len(split['spec_mols']):,})")


## Entrenamiento del decodificador (piloto 50k)

Transformer pequeño desde cero (d_model 256, 4 capas, 8 cabezas),
50,000 moléculas de GEN-TRAIN, 20 épocas, batch 256, lr 3e-4 con
warmup + coseno. Huellas exactas (50%) y corruptas (50%, Tanimoto
objetivo 0.31, imitando el error de la etapa 1). Holdout interno 2%
de GEN-TRAIN (semilla 42) para early stopping y evaluación.


In [ ]:
# Celda 6 — Entrena el piloto desde el paquete de datos, sin modificar scripts.
SCRIPT_ENTRENAR = RUTA_DATOS / "entrenar_decodificador.py"
!python "{SCRIPT_ENTRENAR}" --modo entrenar \
  --datos "{RUTA_DATOS}" --salida "{RUTA_TRABAJO}/salida_piloto" \
  --n_train 50000 --epocas 20 --batch 256 --device auto

In [ ]:
# Celda 7 — Puertas del piloto en holdout (lee metricas.json).
import json
met = json.loads((RUTA_TRABAJO / "salida_piloto" / "metricas.json").read_text())
hf = met["holdout_final"]
print("=== PILOTO B — HOLDOUT ===")
print(f"n_train: {met['n_train']:,} | n_holdout: {met['n_holdout']:,} | "
      f"épocas: {met['epocas']} | vocab: {met['vocab_size']} tokens | "
      f"parámetros: {met['parametros']:,}")
print(f"Reconstrucción exacta (greedy, top-1): {hf['greedy_exact_match']:.4f}")
print(f"Reconstrucción exacta (beam-25):       {hf['beam_exact_match']:.4f}")
print(f"Validez (beam):                        {hf['beam_validez']:.4f}")
print(f"Tanimoto@1 (beam, top-1):              {hf['beam_tanimoto_top1']:.4f}  (piso B2: 0.267)")
print()
print("Regla de corte (propuesta, Elán decide): si la reconstrucción")
print("exacta huella-exacta en holdout es < 0.50, la familia B se corta.")


## Evaluación en GEN-VAL (muestra de 2,000)

El decodificador genera listas beam-25 para una muestra aleatoria de
2,000 moléculas de GEN-VAL (semilla 42) con huellas **exactas** (esto
mide el decodificador solo; la integración con la etapa 1 —huellas
predichas desde el espectro— es la Fase 3). Luego el evaluador del
protocolo puntúa MRR@25, validez, unicidad, novedad y Tanimoto@1.

Para la evaluación completa (13,748 moléculas), cambia `--muestra 0`
en la celda siguiente (la generación beam-25 completa tarda ~1–2 h
en T4; la evaluación con el evaluador, ~35–60 min).


In [ ]:
# Celda 8 — Genera predicciones beam-25 para la muestra GEN-VAL.
!python "{RUTA_DATOS}/entrenar_decodificador.py" --modo predecir \
  --datos "{RUTA_DATOS}" --salida "/content/piloto_b/pred_genval" \
  --checkpoint "/content/piloto_b/salida_piloto/modelo.pt" \
  --lista "{RUTA_DATOS}/verificacion/piloto_b/gen_val_verdad.parquet" \
  --muestra 2000 --beam 25 --huella exacta --device auto


In [ ]:
# Celda 9 — Verdad GEN-VAL en formato del evaluador + evaluación.
# inchikey14 = la clave TAUTOMERIZADA (R-EVAL-6 del protocolo): la
# verdad local se canonicaliza igual que las predicciones.
import pandas as pd
verdad = pd.read_parquet(RUTA_DATOS / "verificacion" / "piloto_b" / "gen_val_verdad.parquet")
print("columnas verdad:", list(verdad.columns))
col_tau = next((c for c in ["inchikey14_tautomerizado", "inchikey14_taut",
                            "inchikey14_tautomerizada"] if c in verdad.columns),
               "inchikey14")
if col_tau == "inchikey14":
    print("AVISO: no se encontró la columna tautomerizada; se usa inchikey14 tal cual.")
verdad_out = pd.DataFrame({
    "molecule_id": verdad["molecule_id"],
    "inchikey14": verdad[col_tau],
    "smiles": verdad["smiles"],
})
verdad_out.to_csv(RUTA_TRABAJO / "verdad_genval.csv", index=False)
print(f"verdad escrita: {len(verdad_out):,} moléculas -> {RUTA_TRABAJO / 'verdad_genval.csv'}")


In [ ]:
# Celda 10 — Evaluador del protocolo sobre la muestra GEN-VAL.
!python "{RUTA_DATOS}/30_evaluar_generativo.py" \
  --predicciones /content/piloto_b/pred_genval/predicciones.csv \
  --verdad /content/piloto_b/verdad_genval.csv \
  --salida /content/piloto_b/evaluacion_genval.json --bootstrap 2000
import json
ev = json.loads((RUTA_TRABAJO / "evaluacion_genval.json").read_text())
print(json.dumps(ev, indent=2, ensure_ascii=False))


In [ ]:
# Celda 11 — Reporte final de puertas + guardado.
import json, shutil
met = json.loads((RUTA_TRABAJO / "salida_piloto" / "metricas.json").read_text())
ev = json.loads((RUTA_TRABAJO / "evaluacion_genval.json").read_text())
col = json.loads((RUTA_DATOS / "verificacion" / "piloto_b" / "colisiones_huella.json").read_text())
hf = met["holdout_final"]
print("=== REPORTE FINAL — PILOTO B ===")
print(f"Techo por colisión de huella (Fase 0): {col}")
print(f"Holdout: exact-match greedy {hf['greedy_exact_match']:.4f} | "
      f"beam-25 {hf['beam_exact_match']:.4f} | validez {hf['beam_validez']:.4f} | "
      f"Tanimoto@1 {hf['beam_tanimoto_top1']:.4f}")
print(f"GEN-VAL (muestra n={ev['filas_evaluadas']:,}): MRR@25 {ev['mrr25']:.4f} "
      f"IC95 {ev['mrr25_ic95_bootstrap']} | top-1 {ev['top1']:.4f} | "
      f"recall@25 {ev['recall25']:.4f}")
print(f"GEN-VAL: validez {ev['validez_media']:.4f} | unicidad {ev['unicidad_media']:.4f} | "
      f"Tanimoto@1 {ev['tanimoto']['top1_media']:.4f} (piso B2 0.267) | "
      f"mejor-en-25 {ev['tanimoto']['mejor25_media']:.4f}")
print()
print("Umbrales del go-ahead: MRR_novel > 0.01 y Tanimoto@1 > 0.267.")
print("Corte propuesto: reconstrucción exacta holdout < 0.50 -> familia B se corta.")
print()
print("Qué reportar a Elán: este reporte + metricas.json + evaluacion_genval.json")
print("+ modelo.pt. Archivos en /content/piloto_b/.")
# Copia de respaldo a Drive (si está montado).
destino = Path("/content/drive/MyDrive/casmi-piloto-b/salida_piloto")
try:
    if Path("/content/drive").exists():
        shutil.copytree(RUTA_TRABAJO, destino, dirs_exist_ok=True)
        print("Respaldo copiado a Drive:", destino)
    else:
        print("Drive no montado: descarga /content/piloto_b/ a mano (Archivo > Descargar).")
except Exception as exc:
    print("No se pudo copiar a Drive:", exc)


## Qué reportar

Copia este reporte y los archivos `metricas.json`,
`evaluacion_genval.json` y `modelo.pt` a Elán (o al chat del
Coordinador). Los números que deciden son:

- **Reconstrucción exacta en holdout** (greedy y beam-25) — puerta de
  la familia B (< 0.50 propuesto → se corta).
- **Tanimoto@1 en GEN-VAL vs piso B2 (0.267)** — puerta de cercanía.
- **MRR@25 en GEN-VAL + IC95** — umbral del go-ahead (> 0.01).

Si el piloto pasa, el siguiente paso es la Fase 1 completa (252k
moléculas) y la integración con la etapa 1 (huellas predichas desde el
espectro, Fase 3). Si no pasa, la familia B se corta y se evalúa la
familia A (transformer espectro→SMILES, con los pesos de Mismetti et
al. como base — ver `verificacion/piloto_b/MS2MOL_VERIFICACION.md`).
